# Baseline tambahan reviewer: adapted QMF (ICML 2023)

Notebook ini menambahkan **satu baseline recent** pada eksperimen MAIN lima-fold yang telah dibekukan. Baseline mengikuti prinsip *Quality-aware Multimodal Fusion* (QMF): bobot tiap modalitas berubah per sampel dan berhubungan negatif dengan ketidakpastian prediksi.

Adaptasi yang dilakukan harus dilaporkan apa adanya:

- tiga expert: rerata tiga encoder teks, transcript, dan visual;
- probabilitas expert berasal dari run MAIN yang sudah selesai;
- kalibrator dan quality weights dilatih hanya pada `meta_fit`;
- early stopping memakai internal group split dari `meta_fit`;
- `calibration` hanya memilih threshold; `test` tidak dipakai untuk tuning;
- modalitas transcript/visual yang tidak tersedia dikeluarkan dari softmax weights;
- ini adaptasi decision-level pada frozen outputs, **bukan reproduksi identik** eksperimen QMF asli.

Rujukan: Zhang et al., “Provable Dynamic Fusion for Low-Quality Multimodal Data,” ICML 2023. Paper: https://proceedings.mlr.press/v202/zhang23ar.html — kode resmi: https://github.com/QingyangZhang/QMF


In [1]:
# 1. Mount Google Drive dan konfigurasi path
try:
    from google.colab import drive
except ImportError:
    print("Runtime lokal terdeteksi; mount Drive dilewati.")
else:
    drive.mount("/content/drive")

from pathlib import Path
import os

RUN_NAME = "reviewer_v12_20265495_fix1"
EXPECTED_RUN_SIGNATURE = "4e3c48bcea1140550b53bac918a92ef7b4b06b9eb8c49e2cd96406218923f6a4"

# Biasanya tidak perlu diubah. Untuk path khusus, isi string absolut di bawah,
# atau set environment variable QMF_RUN_DIR.
RUN_DIR_OVERRIDE = None

KNOWN_PROJECT_ROOTS = [
    Path("/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3"),
    Path("/content/drive/MyDrive/dataset_multimodal_tesis/Perundungan Siber v3"),
]


Mounted at /content/drive


In [2]:
# 2. Import dan konfigurasi training ringan
import copy
import hashlib
import json
import math
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize
from scipy.special import expit, softmax
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit

CONFIG = {
    "baseline": "qmf2023_adapted_decision_level",
    "publication": "Zhang et al., ICML 2023",
    "paper_url": "https://proceedings.mlr.press/v202/zhang23ar.html",
    "official_code_url": "https://github.com/QingyangZhang/QMF",
    "adaptation": (
        "QMF uncertainty-aware dynamic weighting on frozen MAIN probabilities; "
        "text expert is the mean of three text encoders; transcript and visual "
        "experts are availability-masked. This is not an exact reproduction of "
        "the original dataset/backbone experiments."
    ),
    "seeds": [42, 43, 44],
    "max_iterations": 200,
    "patience": 25,
    "weight_decay": 1e-4,
    "internal_validation_fraction": 0.20,
    "internal_split_seed_base": 20265495,
    "lambda_unimodal": 0.20,
    "lambda_rank": 0.10,
    "gradient_clip": 1.0,
    "min_delta": 1e-5,
    "max_rank_pairs_per_modality": 4096,
    "optimizer": "scipy_L-BFGS-B",
    "threshold_grid": [round(x / 100, 2) for x in range(5, 96)],
    "bootstrap_repetitions": 2000,
    "permutation_repetitions": 2000,
    "primary_seed": 42,
}

print("NumPy/SciPy/sklearn:", np.__version__, scipy.__version__, sklearn.__version__)
print("Baseline berjalan di CPU; GPU dan instalasi package tambahan tidak diperlukan.")


NumPy/SciPy/sklearn: 2.1.3 1.16.3 1.6.1
Baseline berjalan di CPU; GPU dan instalasi package tambahan tidak diperlukan.


In [3]:
# 3. Temukan dan audit run MAIN — berhenti keras bila path atau isi salah
def locate_run_dir():
    override = os.environ.get("QMF_RUN_DIR") or RUN_DIR_OVERRIDE
    candidates = []
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend(root / "revision_runs" / RUN_NAME for root in KNOWN_PROJECT_ROOTS)
    shortcut_root = Path("/content/drive/.shortcut-targets-by-id")
    if shortcut_root.exists():
        candidates.extend(shortcut_root.glob(f"*/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/{RUN_NAME}"))
    mydrive = Path("/content/drive/MyDrive")
    if mydrive.exists():
        candidates.extend(mydrive.glob(f"**/revision_runs/{RUN_NAME}"))
    valid = []
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            continue
        if p.is_dir() and (p / "run_manifest.json").is_file():
            valid.append(p)
    valid = list(dict.fromkeys(valid))
    if not valid:
        raise FileNotFoundError(
            "Folder run MAIN tidak ditemukan. Isi RUN_DIR_OVERRIDE dengan path folder "
            f"revision_runs/{RUN_NAME}, bukan folder LOPO."
        )
    if len(valid) > 1 and not override:
        raise RuntimeError(f"Ditemukan lebih dari satu run. Isi RUN_DIR_OVERRIDE: {valid}")
    return valid[0]


RUN_DIR = locate_run_dir()
OUT_DIR = RUN_DIR / "qmf2023_adapted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

required_top = [
    "run_manifest.json", "splits.json", "partition_assignments.csv",
    "modality_masks.csv", "outer_predictions.csv", "TRAINING_COMPLETE.json",
]
missing = [name for name in required_top if not (RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Run MAIN belum lengkap; berkas hilang: {missing}")

manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
if EXPECTED_RUN_SIGNATURE and manifest.get("signature") != EXPECTED_RUN_SIGNATURE:
    raise ValueError(
        "Signature run berbeda. Notebook menolak mencampur hasil eksperimen lain. "
        f"Ditemukan={manifest.get('signature')}"
    )
if manifest.get("lopo", False):
    raise ValueError("Path menunjuk run LOPO. Baseline reviewer harus memakai run MAIN.")

splits = json.loads((RUN_DIR / "splits.json").read_text())
masks_df = pd.read_csv(RUN_DIR / "modality_masks.csv")
assign_df = pd.read_csv(RUN_DIR / "partition_assignments.csv")
outer = pd.read_csv(RUN_DIR / "outer_predictions.csv")

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map({"true": True, "false": False, "1": True, "0": False})

for col in ["audio_effective", "visual_effective"]:
    masks_df[col] = as_bool(masks_df[col])
for col in ["is_augmented", "audio_effective", "visual_effective"]:
    outer[col] = as_bool(outer[col])

if masks_df.sample_id.duplicated().any() or outer.sample_id.duplicated().any():
    raise ValueError("sample_id harus unik pada masks dan outer predictions.")
if set(masks_df.sample_id) != set(outer.sample_id):
    raise ValueError("Cakupan sample_id masks dan outer predictions berbeda.")
if outer.label.isna().any() or not set(outer.label.astype(int).unique()).issubset({0, 1}):
    raise ValueError("Label biner pada outer_predictions tidak valid.")

lookup = outer.set_index("sample_id", drop=False)
mask_lookup = masks_df.set_index("sample_id", drop=False)
role_order = ["fit", "early_stop", "meta_fit", "calibration", "test"]
for fold, roles in splits.items():
    if set(roles) != set(role_order):
        raise ValueError(f"Role {fold} tidak sesuai protokol MAIN: {list(roles)}")
    role_groups = []
    for role in role_order:
        expected_ids = masks_df.iloc[np.asarray(roles[role], dtype=int)].sample_id.tolist()
        assigned = assign_df.loc[
            (assign_df.experiment == fold) & (assign_df.role == role), "sample_id"
        ].tolist()
        if expected_ids != assigned:
            raise ValueError(f"Urutan assignment tidak cocok: {fold}/{role}")
        role_groups.append(set(assign_df.loc[
            (assign_df.experiment == fold) & (assign_df.role == role), "group_id"
        ]))
    for i in range(len(role_groups)):
        for j in range(i + 1, len(role_groups)):
            if role_groups[i] & role_groups[j]:
                raise ValueError(f"Kebocoran group pada {fold}: {role_order[i]} vs {role_order[j]}")
    for role in ["meta_fit", "calibration", "test"]:
        p = RUN_DIR / fold / f"base_probabilities_{role}.csv"
        if not p.is_file():
            raise FileNotFoundError(f"Berkas probabilitas hilang: {p}")
        d = pd.read_csv(p)
        expected_ids = masks_df.iloc[np.asarray(roles[role], dtype=int)].sample_id.tolist()
        if d.sample_id.tolist() != expected_ids:
            raise ValueError(f"Urutan probabilitas tidak cocok: {fold}/{role}")
        needed = {"indobert", "indobertweet", "mbert", "transcript", "visual"}
        if not needed.issubset(d.columns):
            raise ValueError(f"Kolom probabilitas kurang pada {fold}/{role}")
        values = d[list(needed)].to_numpy(float)
        if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
            raise ValueError(f"Probabilitas invalid pada {fold}/{role}")

print("Run MAIN:", RUN_DIR)
print("Signature:", manifest["signature"])
print("Fold:", list(splits))
print("Observasi unik:", len(outer), "| audio efektif:", int(masks_df.audio_effective.sum()), "| visual efektif:", int(masks_df.visual_effective.sum()))
print("Audit input lulus; test tidak akan digunakan untuk fitting atau threshold selection.")


Run MAIN: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1
Signature: 4e3c48bcea1140550b53bac918a92ef7b4b06b9eb8c49e2cd96406218923f6a4
Fold: ['fold_1', 'fold_2', 'fold_3', 'fold_4', 'fold_5']
Observasi unik: 15044 | audio efektif: 450 | visual efektif: 565
Audit input lulus; test tidak akan digunakan untuk fitting atau threshold selection.


In [4]:
# 4. Implementasi adapted QMF dan fungsi evaluasi
MODALITIES = ["text_mean", "transcript", "visual"]

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)


def atomic_json(path, obj):
    path = Path(path)
    tmp = Path("/tmp") / f"{path.name}.{os.getpid()}.tmp"
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tmp, path)
    tmp.unlink(missing_ok=True)


def atomic_npz(path, **arrays):
    path = Path(path)
    tmp = Path("/tmp") / f"{path.stem}.{os.getpid()}.npz"
    np.savez_compressed(tmp, **arrays)
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(tmp, path)
    tmp.unlink(missing_ok=True)


def load_role(fold, role):
    d = pd.read_csv(RUN_DIR / fold / f"base_probabilities_{role}.csv")
    ids = d.sample_id.tolist()
    meta = lookup.loc[ids]
    mm = mask_lookup.loc[ids]
    text_mean = d[["indobert", "indobertweet", "mbert"]].mean(axis=1).to_numpy(np.float32)
    probs = np.column_stack([
        text_mean,
        d.transcript.to_numpy(np.float32),
        d.visual.to_numpy(np.float32),
    ]).astype(np.float32)
    available = np.column_stack([
        np.ones(len(d), dtype=np.float32),
        mm.audio_effective.to_numpy(np.float32),
        mm.visual_effective.to_numpy(np.float32),
    ])
    return {
        "ids": np.asarray(ids),
        "probs": probs,
        "available": available,
        "label": meta.label.to_numpy(np.int64),
        "group": meta.group_id.astype(str).to_numpy(),
    }


def softplus(x):
    return np.logaddexp(0.0, x)


def forward_qmf(theta, probs, available):
    """QMF-style uncertainty-aware dynamic decision fusion.

    The 12 fitted parameters are three monotone calibration scales, three
    calibration biases, three negative uncertainty slopes and three nonnegative
    quality intercepts. Missing experts receive exactly zero normalized weight.
    """
    theta = np.asarray(theta, dtype=np.float64)
    if theta.shape != (12,) or not np.isfinite(theta).all():
        raise ValueError("Parameter QMF invalid.")
    eps = 1e-5
    p = np.clip(np.asarray(probs, dtype=np.float64), eps, 1 - eps)
    available = np.asarray(available, dtype=np.float64)
    base_logits = np.log(p) - np.log1p(-p)
    scale = softplus(theta[0:3]) + 1e-3
    bias = theta[3:6]
    alpha = softplus(theta[6:9]) + 1e-3
    beta = softplus(theta[9:12])
    modal_logits = base_logits * scale + bias
    modal_probs = np.clip(expit(modal_logits), eps, 1 - eps)
    entropy = -(
        modal_probs * np.log(modal_probs)
        + (1 - modal_probs) * np.log1p(-modal_probs)
    ) / math.log(2.0)
    quality = beta - alpha * entropy
    quality = np.where(available > 0.5, quality, -1e4)
    weights = softmax(quality, axis=1)
    fused_logits = np.sum(weights * modal_logits, axis=1)
    return fused_logits, modal_logits, weights, entropy


def binary_log_loss_from_logits(logits, labels):
    labels = np.asarray(labels, dtype=np.float64)
    return np.logaddexp(0.0, logits) - labels * logits


def make_rank_pairs(available, seed):
    rng = np.random.default_rng(seed)
    pairs = []
    for m in range(available.shape[1]):
        ids = np.flatnonzero(available[:, m] > 0.5)
        n = min(CONFIG["max_rank_pairs_per_modality"], max(len(ids) * 2, 2))
        if len(ids) < 2:
            pairs.append((np.array([], dtype=int), np.array([], dtype=int)))
        else:
            pairs.append((rng.choice(ids, n, replace=True), rng.choice(ids, n, replace=True)))
    return pairs


def objective(theta, probs, available, labels, rank_pairs, return_parts=False):
    fused, modal_logits, weights, _ = forward_qmf(theta, probs, available)
    fusion_loss = float(binary_log_loss_from_logits(fused, labels).mean())
    modal_target = np.asarray(labels)[:, None]
    per_modal = binary_log_loss_from_logits(modal_logits, modal_target)
    unimodal_loss = float((per_modal * available).sum() / max(available.sum(), 1.0))
    rank_terms = []
    for m, (left, right) in enumerate(rank_pairs):
        if len(left) == 0:
            continue
        delta_loss = per_modal[left, m] - per_modal[right, m]
        delta_weight = weights[left, m] - weights[right, m]
        rank_terms.append(float(np.maximum(delta_loss * delta_weight, 0.0).mean()))
    rank_loss = float(np.mean(rank_terms)) if rank_terms else 0.0
    l2 = float(CONFIG["weight_decay"] * np.dot(theta, theta))
    total = fusion_loss + CONFIG["lambda_unimodal"] * unimodal_loss + CONFIG["lambda_rank"] * rank_loss + l2
    if return_parts:
        return total, fusion_loss, unimodal_loss, rank_loss, l2
    return total


def predict(theta, probs, available):
    logits, _, weights, _ = forward_qmf(theta, probs, available)
    return expit(logits).astype(np.float64), weights.astype(np.float64)


def train_one(fold, fold_index, seed, meta, cal, test):
    run_out = OUT_DIR / fold / f"seed_{seed}"
    run_out.mkdir(parents=True, exist_ok=True)
    cache = run_out / "predictions.npz"
    cache_support = [
        run_out / "best_parameters.npz",
        run_out / "fit_summary.json",
        run_out / "history.json",
    ]
    if cache.exists() and all(p.exists() for p in cache_support):
        with np.load(cache, allow_pickle=False) as z:
            if z["cal"].shape == (len(cal["ids"]),) and z["test"].shape == (len(test["ids"]),):
                print(f"{fold} seed {seed}: cache valid")
                return z["cal"], z["test"], z["cal_weights"], z["test_weights"]

    split_seed = CONFIG["internal_split_seed_base"] + fold_index
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=CONFIG["internal_validation_fraction"],
        random_state=split_seed,
    )
    train_idx, val_idx = next(splitter.split(meta["probs"], meta["label"], meta["group"]))
    train_groups = set(meta["group"][train_idx])
    val_groups = set(meta["group"][val_idx])
    if train_groups & val_groups:
        raise RuntimeError("Internal meta_fit group leakage.")
    if len(np.unique(meta["label"][train_idx])) != 2 or len(np.unique(meta["label"][val_idx])) != 2:
        raise RuntimeError("Internal group split kehilangan salah satu kelas.")

    seed_all(seed)
    rng = np.random.default_rng(seed)
    initial = rng.normal(0.0, 0.02, size=12)
    rank_pairs = make_rank_pairs(meta["available"][train_idx], seed + 1000 * fold_index)
    best_loss = float("inf")
    best_theta = initial.copy()
    bad_epochs = 0
    history = []

    class EarlyStop(Exception):
        pass

    def callback(theta):
        nonlocal best_loss, best_theta, bad_epochs
        parts = objective(
            theta,
            meta["probs"][train_idx],
            meta["available"][train_idx],
            meta["label"][train_idx],
            rank_pairs,
            return_parts=True,
        )
        val_logits, _, _, _ = forward_qmf(
            theta, meta["probs"][val_idx], meta["available"][val_idx]
        )
        val_loss = float(binary_log_loss_from_logits(val_logits, meta["label"][val_idx]).mean())
        iteration = len(history) + 1
        history.append({
            "iteration": iteration,
            "total_loss": parts[0],
            "fusion_loss": parts[1],
            "unimodal_loss": parts[2],
            "rank_loss": parts[3],
            "l2_penalty": parts[4],
            "internal_group_validation_bce": val_loss,
        })
        if val_loss < best_loss - CONFIG["min_delta"]:
            best_loss = val_loss
            best_theta = np.asarray(theta).copy()
            bad_epochs = 0
        else:
            bad_epochs += 1
        if iteration == 1 or iteration % 25 == 0:
            print(f"{fold} seed {seed} iteration {iteration}: val BCE={val_loss:.5f}")
        if bad_epochs >= CONFIG["patience"]:
            raise EarlyStop()

    try:
        result = minimize(
            objective,
            initial,
            args=(
                meta["probs"][train_idx], meta["available"][train_idx],
                meta["label"][train_idx], rank_pairs, False,
            ),
            method="L-BFGS-B",
            callback=callback,
            options={"maxiter": CONFIG["max_iterations"], "ftol": 1e-10, "maxls": 30},
        )
        optimizer_status = {"success": bool(result.success), "message": str(result.message), "nit": int(result.nit)}
        if not history:
            callback(result.x)
    except EarlyStop:
        optimizer_status = {"success": True, "message": "early stopping", "nit": len(history)}

    if not np.isfinite(best_theta).all() or not np.isfinite(best_loss):
        raise RuntimeError("Tidak ada parameter valid.")
    atomic_npz(run_out / "best_parameters.npz", theta=best_theta)
    atomic_json(run_out / "fit_summary.json", {
        "config": CONFIG,
        "modalities": MODALITIES,
        "fold": fold,
        "seed": seed,
        "best_internal_validation_bce": best_loss,
        "iterations_ran": len(history),
        "train_groups": len(train_groups),
        "validation_groups": len(val_groups),
        "optimizer_status": optimizer_status,
    })
    atomic_json(run_out / "history.json", history)

    p_cal, w_cal = predict(best_theta, cal["probs"], cal["available"])
    p_test, w_test = predict(best_theta, test["probs"], test["available"])
    if not np.isfinite(p_cal).all() or not np.isfinite(p_test).all():
        raise ValueError("Prediksi QMF mengandung nilai non-finite.")
    atomic_npz(cache, cal=p_cal, test=p_test, cal_weights=w_cal, test_weights=w_test)
    return p_cal, p_test, w_cal, w_test


def select_threshold(y, p):
    grid = np.asarray(CONFIG["threshold_grid"], dtype=float)
    scores = np.asarray([
        f1_score(y, p >= t, average="macro", labels=[0, 1], zero_division=0)
        for t in grid
    ])
    best = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
    j = min(best, key=lambda k: (abs(grid[k] - 0.5), grid[k]))
    return float(grid[j]), scores.tolist()


def metric_row(y, p, threshold):
    pred = p >= threshold
    return {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "macro_precision": float(precision_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),
        "macro_recall": float(recall_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),
        "macro_f1": float(f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0)),
        "roc_auc": float(roc_auc_score(y, p)),
        "average_precision": float(average_precision_score(y, p)),
    }


In [5]:
# 5. Jalankan lima fold × tiga seed — aman di-resume dari cache
start_time = time.time()
prediction_rows = []
threshold_records = []
weight_records = []

atomic_json(OUT_DIR / "configuration.json", {
    **CONFIG,
    "run_name": RUN_NAME,
    "run_signature": manifest["signature"],
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "sklearn_version": sklearn.__version__,
    "device_used": "CPU",
    "modalities": MODALITIES,
})

for fold_index, fold in enumerate(splits):
    print("\n" + "=" * 70)
    print(fold.upper())
    meta = load_role(fold, "meta_fit")
    cal = load_role(fold, "calibration")
    test = load_role(fold, "test")
    for seed in CONFIG["seeds"]:
        p_cal, p_test, w_cal, w_test = train_one(fold, fold_index, seed, meta, cal, test)
        threshold, grid_scores = select_threshold(cal["label"], p_cal)
        threshold_records.append({
            "experiment": fold,
            "seed": seed,
            "model": f"qmf2023_adapted_seed{seed}",
            "threshold": threshold,
            "calibration_macro_f1": max(grid_scores),
            "threshold_grid_scores": grid_scores,
        })
        for i, sample_id in enumerate(test["ids"]):
            prediction_rows.append({
                "sample_id": sample_id,
                "experiment": fold,
                "seed": seed,
                "p_qmf2023_adapted": float(p_test[i]),
                "t_qmf2023_adapted": threshold,
                "pred_qmf2023_adapted": int(p_test[i] >= threshold),
            })
        mean_weights = w_test.mean(axis=0)
        weight_records.append({
            "experiment": fold,
            "seed": seed,
            **{f"mean_weight_{name}": float(mean_weights[j]) for j, name in enumerate(MODALITIES)},
            "test_audio_available": int(test["available"][:, 1].sum()),
            "test_visual_available": int(test["available"][:, 2].sum()),
        })
        print(f"{fold} seed {seed}: threshold={threshold:.2f}, cal macro-F1={max(grid_scores):.4f}")

pred_long = pd.DataFrame(prediction_rows)
if pred_long.duplicated(["sample_id", "seed"]).any():
    raise ValueError("Prediksi test ganda.")
if len(pred_long) != len(outer) * len(CONFIG["seeds"]):
    raise ValueError("Jumlah prediksi aggregate tidak lengkap.")

pd.DataFrame(threshold_records).to_json(OUT_DIR / "thresholds.json", orient="records", indent=2)
pd.DataFrame(weight_records).to_csv(OUT_DIR / "mean_dynamic_weights.csv", index=False)
pred_long.to_csv(OUT_DIR / "qmf_test_predictions_long.csv", index=False)
print(f"Training/prediction selesai dalam {(time.time() - start_time) / 60:.2f} menit.")



FOLD_1
fold_1 seed 42 iteration 1: val BCE=0.33074
fold_1 seed 42 iteration 25: val BCE=0.30480
fold_1 seed 42: threshold=0.41, cal macro-F1=0.8928
fold_1 seed 43 iteration 1: val BCE=0.32953
fold_1 seed 43 iteration 25: val BCE=0.30491
fold_1 seed 43: threshold=0.42, cal macro-F1=0.8922
fold_1 seed 44 iteration 1: val BCE=0.32942
fold_1 seed 44 iteration 25: val BCE=0.30481
fold_1 seed 44: threshold=0.41, cal macro-F1=0.8922

FOLD_2
fold_2 seed 42 iteration 1: val BCE=0.29747
fold_2 seed 42 iteration 25: val BCE=0.25448
fold_2 seed 42: threshold=0.55, cal macro-F1=0.8972
fold_2 seed 43 iteration 1: val BCE=0.29638
fold_2 seed 43 iteration 25: val BCE=0.25451
fold_2 seed 43: threshold=0.56, cal macro-F1=0.8967
fold_2 seed 44 iteration 1: val BCE=0.29585
fold_2 seed 44 iteration 25: val BCE=0.25444
fold_2 seed 44: threshold=0.55, cal macro-F1=0.8972

FOLD_3
fold_3 seed 42 iteration 1: val BCE=0.33703
fold_3 seed 42 iteration 25: val BCE=0.29887
fold_3 seed 42: threshold=0.47, cal macro

In [6]:
# 6. Regenerasi metrik, stabilitas seed, dan paired uncertainty
core_cols = [
    "sample_id", "group_id", "platform", "label", "label_source", "is_augmented",
    "audio_effective", "visual_effective", "experiment",
    "p_full_lr", "t_full_lr", "pred_full_lr",
]
wide = outer[core_cols].copy()
for seed in CONFIG["seeds"]:
    d = pred_long.loc[pred_long.seed == seed].drop(columns=["experiment", "seed"])
    d = d.rename(columns={
        "p_qmf2023_adapted": f"p_qmf2023_adapted_seed{seed}",
        "t_qmf2023_adapted": f"t_qmf2023_adapted_seed{seed}",
        "pred_qmf2023_adapted": f"pred_qmf2023_adapted_seed{seed}",
    })
    wide = wide.merge(d, on="sample_id", how="left", validate="one_to_one")
if wide.filter(regex=r"^(p|t|pred)_qmf").isna().any().any():
    raise ValueError("Ada prediksi QMF yang tidak terpasang ke outer observations.")

wide.to_csv(OUT_DIR / "outer_predictions_with_qmf2023.csv", index=False)

metric_records = []
fold_metric_records = []
for seed in CONFIG["seeds"]:
    pc = f"p_qmf2023_adapted_seed{seed}"
    tc = f"t_qmf2023_adapted_seed{seed}"
    for subset, data in [("all", wide), ("non_augmented", wide.loc[~wide.is_augmented])]:
        # threshold berbeda per fold dan sudah direkam per baris
        pred = data[pc].to_numpy() >= data[tc].to_numpy()
        y = data.label.to_numpy(int)
        p = data[pc].to_numpy(float)
        metric_records.append({
            "subset": subset,
            "model": f"qmf2023_adapted_seed{seed}",
            "seed": seed,
            "n": len(data),
            "accuracy": accuracy_score(y, pred),
            "macro_precision": precision_score(y, pred, average="macro", labels=[0, 1], zero_division=0),
            "macro_recall": recall_score(y, pred, average="macro", labels=[0, 1], zero_division=0),
            "macro_f1": f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0),
            "roc_auc": roc_auc_score(y, p),
            "average_precision": average_precision_score(y, p),
        })
    for fold, data in wide.loc[~wide.is_augmented].groupby("experiment", sort=True):
        pred = data[pc].to_numpy() >= data[tc].to_numpy()
        y = data.label.to_numpy(int)
        p = data[pc].to_numpy(float)
        fold_metric_records.append({
            "experiment": fold,
            "seed": seed,
            "n": len(data),
            "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro", labels=[0, 1], zero_division=0),
            "roc_auc": roc_auc_score(y, p),
            "average_precision": average_precision_score(y, p),
        })

metrics_df = pd.DataFrame(metric_records)
fold_metrics_df = pd.DataFrame(fold_metric_records)
metrics_df.to_csv(OUT_DIR / "table_qmf_metrics.csv", index=False)
fold_metrics_df.to_csv(OUT_DIR / "qmf_fold_metrics.csv", index=False)

primary = metrics_df.loc[metrics_df.subset == "non_augmented"].copy()
stability = {
    "model": "qmf2023_adapted",
    "seeds": CONFIG["seeds"],
    "n": int(primary.n.iloc[0]),
    "macro_f1_mean": float(primary.macro_f1.mean()),
    "macro_f1_sample_sd": float(primary.macro_f1.std(ddof=1)),
    "accuracy_mean": float(primary.accuracy.mean()),
    "accuracy_sample_sd": float(primary.accuracy.std(ddof=1)),
    "roc_auc_mean": float(primary.roc_auc.mean()),
    "average_precision_mean": float(primary.average_precision.mean()),
}
atomic_json(OUT_DIR / "seed_stability.json", stability)

def grouped_paired_uncertainty(data, seed, repetitions, random_seed=20265495):
    y = data.label.to_numpy(int)
    pred_new = data[f"pred_qmf2023_adapted_seed{seed}"].to_numpy(int)
    pred_ref = data.pred_full_lr.to_numpy(int)
    groups, group_code = np.unique(data.group_id.astype(str).to_numpy(), return_inverse=True)
    def delta(sample_weight=None, a=pred_new, b=pred_ref):
        fa = f1_score(y, a, average="macro", labels=[0, 1], zero_division=0, sample_weight=sample_weight)
        fb = f1_score(y, b, average="macro", labels=[0, 1], zero_division=0, sample_weight=sample_weight)
        return float(fa - fb)
    point = delta()
    rng = np.random.default_rng(random_seed + seed)
    boot = np.empty(repetitions)
    for b in range(repetitions):
        drawn = rng.integers(0, len(groups), size=len(groups))
        group_weights = np.bincount(drawn, minlength=len(groups))
        boot[b] = delta(sample_weight=group_weights[group_code])
    perm = np.empty(repetitions)
    for b in range(repetitions):
        swap_group = rng.integers(0, 2, size=len(groups)).astype(bool)
        swap = swap_group[group_code]
        a = np.where(swap, pred_ref, pred_new)
        c = np.where(swap, pred_new, pred_ref)
        perm[b] = delta(a=a, b=c)
    return {
        "comparison": f"qmf2023_adapted_seed{seed}_minus_full_lr",
        "subset": "non_augmented",
        "n": int(len(data)),
        "groups": int(len(groups)),
        "macro_f1_difference": point,
        "grouped_bootstrap_95_ci": [float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))],
        "grouped_sign_permutation_two_sided_p": float((1 + np.sum(np.abs(perm) >= abs(point))) / (repetitions + 1)),
        "bootstrap_repetitions": repetitions,
        "permutation_repetitions": repetitions,
        "qmf_right_full_wrong": int(np.sum((pred_new == y) & (pred_ref != y))),
        "qmf_wrong_full_right": int(np.sum((pred_new != y) & (pred_ref == y))),
        "disagreements": int(np.sum(pred_new != pred_ref)),
    }

paired = grouped_paired_uncertainty(
    wide.loc[~wide.is_augmented].reset_index(drop=True),
    CONFIG["primary_seed"],
    CONFIG["bootstrap_repetitions"],
)
atomic_json(OUT_DIR / "paired_qmf_vs_full_lr.json", paired)

summary_columns = ["model", "n", "accuracy", "macro_f1", "roc_auc", "average_precision"]
try:
    display(primary[summary_columns])
except NameError:
    print(primary[summary_columns].to_string(index=False))
print("Seed stability:", stability)
print("Paired primary seed vs full LR:", paired)


,model,n,accuracy,macro_f1,roc_auc,average_precision
1,qmf2023_adapted_seed42,13307,0.888029,0.878093,0.954893,0.914145
3,qmf2023_adapted_seed43,13307,0.887803,0.877865,0.954831,0.914065
5,qmf2023_adapted_seed44,13307,0.887954,0.878028,0.954913,0.914258


Seed stability: {'model': 'qmf2023_adapted', 'seeds': [42, 43, 44], 'n': 13307, 'macro_f1_mean': 0.8779952705443189, 'macro_f1_sample_sd': 0.00011768437590516888, 'accuracy_mean': 0.887928659101726, 'accuracy_sample_sd': 0.00011479110480588534, 'roc_auc_mean': 0.954879020936236, 'average_precision_mean': 0.91415579434053}
Paired primary seed vs full LR: {'comparison': 'qmf2023_adapted_seed42_minus_full_lr', 'subset': 'non_augmented', 'n': 13307, 'groups': 3585, 'macro_f1_difference': -0.0030627756723160537, 'grouped_bootstrap_95_ci': [-0.00594330856017645, -0.0001586657667007796], 'grouped_sign_permutation_two_sided_p': 0.04447776111944028, 'bootstrap_repetitions': 2000, 'permutation_repetitions': 2000, 'qmf_right_full_wrong': 140, 'qmf_wrong_full_right': 179, 'disagreements': 319}


In [7]:
# 7. Audit akhir dan ZIP yang harus dikirim kembali
readme = f"""# QMF 2023 adapted baseline results

Run MAIN signature: `{manifest['signature']}`

Method: adapted decision-level Quality-aware Multimodal Fusion (Zhang et al., ICML 2023).
Frozen experts: mean of IndoBERT/IndoBERTweet/mBERT probabilities, transcript probability, and visual probability.
Transcript and visual experts are excluded when their effective-availability masks are false.

Protocol:
- five frozen outer folds from the MAIN run;
- fit and internal group-wise early stopping only within meta_fit;
- threshold selected only on calibration over 0.05--0.95;
- outer test untouched by training and threshold selection;
- seeds 42, 43, 44;
- primary reporting excludes augmented observations;
- grouped bootstrap and grouped sign permutation compare primary seed 42 with full LR.

This is a controlled adaptation on saved decision-level outputs, not an exact reproduction of the original QMF backbones or datasets.
"""
(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

expected_outputs = [
    "configuration.json", "thresholds.json", "mean_dynamic_weights.csv",
    "qmf_test_predictions_long.csv", "outer_predictions_with_qmf2023.csv",
    "table_qmf_metrics.csv", "qmf_fold_metrics.csv", "seed_stability.json",
    "paired_qmf_vs_full_lr.json", "README.md",
]
missing = [x for x in expected_outputs if not (OUT_DIR / x).is_file()]
if missing:
    raise FileNotFoundError(f"Output akhir belum lengkap: {missing}")
for fold in splits:
    for seed in CONFIG["seeds"]:
        for name in ["best_parameters.npz", "fit_summary.json", "history.json", "predictions.npz"]:
            p = OUT_DIR / fold / f"seed_{seed}" / name
            if not p.is_file():
                raise FileNotFoundError(f"Checkpoint/output hilang: {p}")

hashes = {
    str(p.relative_to(OUT_DIR)): sha256(p)
    for p in sorted(OUT_DIR.rglob("*")) if p.is_file() and p.name != "FILE_HASHES.json"
}
atomic_json(OUT_DIR / "FILE_HASHES.json", hashes)
atomic_json(OUT_DIR / "QMF2023_COMPLETE.json", {
    "status": "complete",
    "run_signature": manifest["signature"],
    "folds": list(splits),
    "seeds": CONFIG["seeds"],
    "prediction_rows": int(len(pred_long)),
    "unique_test_sample_ids": int(pred_long.sample_id.nunique()),
})

ZIP_PATH = RUN_DIR / "QMF2023_ADAPTED_RESULTS_FOR_REVIEW.zip"
tmp_zip = Path("/tmp") / ZIP_PATH.name
with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, Path(OUT_DIR.name) / p.relative_to(OUT_DIR))
with zipfile.ZipFile(tmp_zip, "r") as zf:
    bad = zf.testzip()
    if bad is not None:
        raise IOError(f"ZIP rusak pada: {bad}")
shutil.copy2(tmp_zip, ZIP_PATH)
tmp_zip.unlink(missing_ok=True)

print("\nSELESAI.")
print("ZIP hasil:", ZIP_PATH)
print("Ukuran ZIP:", round(ZIP_PATH.stat().st_size / 1024 / 1024, 2), "MB")
print("Kirim kembali: (1) ZIP di atas dan (2) notebook ini setelah semua sel selesai dijalankan.")



SELESAI.
ZIP hasil: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1/QMF2023_ADAPTED_RESULTS_FOR_REVIEW.zip
Ukuran ZIP: 1.87 MB
Kirim kembali: (1) ZIP di atas dan (2) notebook ini setelah semua sel selesai dijalankan.
